# Fine-tune GenerativeTerrain on your own terrain

This notebook adapts the **pretrained base model** to *your* terrain as a new
**style** — without retraining the base and without damaging it (the base
weights are frozen; a unit test guarantees they come out bit-identical).

**What you need:**

1. A base checkpoint directory (`artifacts/base_v1/`), produced by the last
   step of `GenerativeTerrain.ipynb` — or shipped to you by someone else.
2. A folder of chunk CSVs exported from **your** world with the plugin's
   `/grabchunkarea` (aim for 50+ chunks of the terrain whose style you want;
   more is better).

**What you get:** one exported ONNX model that generates both the vanilla
base style and your custom style — in game, `/generateterrain` uses the base
and `/generateterrain style <name>` uses yours.

**How it works:** the decoder is conditioned on a style embedding through
FiLM layers (per-channel scale/shift at every U-Net level and in the surface
heightmap head). Fine-tuning trains the new style vector and those small
projections; the convolutions, the encoder, and the base style row stay
frozen. `config.ft_unfreeze` controls the size of the adaptation:

| `ft_unfreeze` | Trains | Use when |
|---|---|---|
| `"style"` | style vector only | your terrain is close to vanilla |
| `"film"` *(default)* | + FiLM projections | most cases |
| `"decoder"` | + whole decoder & heightmap head | 100+ chunks, very different terrain |

In [ ]:
# --- Make the gt_terrain package importable regardless of where Jupyter started.
import sys
from pathlib import Path

ML_DIR = Path.cwd()
while ML_DIR.name != "ml" and ML_DIR != ML_DIR.parent:
    ML_DIR = ML_DIR.parent
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from gt_terrain.config import Config
from gt_terrain import data, train, evaluate, generate, export

cfg = Config()

# >>> EDIT THESE THREE <<<
BASE_DIR   = cfg.artifact_dir / "base_v1"        # the frozen base checkpoint
MY_CHUNKS  = r"C:\\path\\to\\my_world_chunks"       # folder of YOUR chunk CSVs
STYLE_NAME = "myworld"                            # how you'll call it in game

print("base checkpoint:", BASE_DIR)
print("custom chunks  :", MY_CHUNKS)
print("style name     :", STYLE_NAME)

## Fine-tune

One call. Architecture and mappings are rebuilt from the base manifest (never
from your local config, so mismatches are impossible). Your chunks are gridded
with the base block grouping; biomes the base never saw map to the trained
`UNKNOWN` row. Training runs with EMA and early stopping and typically takes
minutes, not hours — it is an adapter-sized update, not a retrain.

In [ ]:
res = train.fine_tune(BASE_DIR, MY_CHUNKS, STYLE_NAME, config=cfg)

print()
print("style id :", res.style_id)
print("registry :", res.styles)
evaluate.plot_training_curves(
    res.history, cfg.artifact_dir / f"ft_{STYLE_NAME}_curves.png", f"Fine-tune: {STYLE_NAME}"
)
from IPython.display import Image
Image(str(cfg.artifact_dir / f"ft_{STYLE_NAME}_curves.png"))

## Compare the base style vs your style

Same latent samples, same biome — only the style id differs. The vertical
profile of your style should drift toward your world's block statistics
(e.g. more sand and water for an ocean world, higher surfaces for mountains).

In [ ]:
import numpy as np

# Your chunks, regridded for the comparison plot.
ft_cfg = res.config
my_grids, my_biomes = data.build_dataset(ft_cfg, res.grouping, res.biome_encoder)

biome_id = int(np.bincount(my_biomes).argmax())     # your most common biome
base_samples = generate.sample_grids(res.model, biome_id, n=4, config=ft_cfg, style_id=0)
mine_samples = generate.sample_grids(res.model, biome_id, n=4, config=ft_cfg,
                                     style_id=res.style_id)

print("diversity (base):", round(evaluate.sample_diversity(base_samples), 4))
print("diversity (mine):", round(evaluate.sample_diversity(mine_samples), 4))

evaluate.plot_vertical_profiles(
    my_grids, mine_samples, res.grouping, ft_cfg,
    ft_cfg.artifact_dir / f"ft_{STYLE_NAME}_profile.png",
)
Image(str(ft_cfg.artifact_dir / f"ft_{STYLE_NAME}_profile.png"))

## Export for the plugin

One model, all styles. Copy the four files into
`<server>/plugins/GenerativeTerrain/`, restart the server, then:

```
/generateterrain                  # base style
/generateterrain style myworld    # your style
```

In [ ]:
onnx_path = export.export_decoder_onnx(res.model, res.config)
group_map, biome_map, style_map = export.write_mappings(
    res.grouping, res.biome_encoder, res.config, styles=res.styles
)
out_shape = export.verify_onnx(onnx_path, res.config, res.biome_encoder.num_biomes)

print("ONNX decoder :", onnx_path)
print("output shape :", out_shape)
print("group mapping:", group_map)
print("biome mapping:", biome_map)
print("style mapping:", style_map)
print("\nCopy these into: <server>/plugins/GenerativeTerrain/")

## Tips

* **Not stylised enough?** Collect more chunks of the terrain you want, or set
  `cfg = cfg.with_(ft_unfreeze="decoder", ft_lr=2e-4)` and re-run (needs more
  data to avoid overfitting, but adapts much more).
* **Several styles:** just run again with a different `STYLE_NAME` — up to
  `max_custom_styles` (8 by default) live in one model. Re-using a name
  retrains that slot.
* **Sharing:** send someone your `base_v1/` folder; they can fine-tune their
  own styles on top of it with this notebook alone.